# Exploração raw — movimentação de derivados

Perfil exploratório dos CSVs do SIMP (ZIPs por produto) e das tabelas de logística.

**Pré-requisito:** `py estudos/movimentacao-derivados/pipelines/download_raw.py`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import RAW_DIR, REPO_ROOT, SAMPLE_LIQUIDOS, PRODUCTS


def read_anp_csv(path: Path) -> pd.DataFrame:
    line = path.read_text(encoding="latin-1", errors="replace").splitlines()[0]
    sep = ";" if line.count(";") >= line.count(",") else ","
    return pd.read_csv(path, encoding="latin-1", sep=sep, dtype=str, on_bad_lines="skip")


if not SAMPLE_LIQUIDOS.exists():
    raise FileNotFoundError(f"Execute download_raw.py. Ausente: {SAMPLE_LIQUIDOS}")

df = read_anp_csv(SAMPLE_LIQUIDOS)
print(f"Snapshot: {SAMPLE_LIQUIDOS.relative_to(REPO_ROOT)}")
print(f"Linhas: {len(df):,} | Colunas: {list(df.columns)}")

## 1. Perfil — combustíveis líquidos (vendas atual)

In [ ]:
display(df.head(3))
display(df.describe(include="all").transpose().head(12))

## 2. Sub-bases por produto (ZIPs extraídos)

In [ ]:
rows = []
for product in PRODUCTS:
    files = sorted((RAW_DIR / product).glob("*.csv"))
    rows.append({"produto": product, "arquivos_csv": len(files)})
pd.DataFrame(rows)

## 3. Inventário empírico (todos os CSVs)

Gerar tabela completa:

```bash
py estudos/movimentacao-derivados/export_inventario_raw.py
```